# NB13 — RQ2 Forecasting Benchmark

NB12 established that the DH lead-lag signal doesn't survive multiplicity-corrected inference under the circular-shift null (0/12 BH survivors). NB13 asks a different, decision-relevant question: even setting aside significance testing, does knowing a country's weapon-class acquisitions genuinely improve **genuine out-of-sample forecasts** of next-year conflict-participation changes, beyond what a naive/autoregressive baseline already gets you?

This is a **forecasting-skill** question, not an inference question. The headline result is reported as RMSE improvement / skill score — not as a significance claim — throughout this notebook.

## Section 0 — Setup

In [1]:
import sys, os, time
from pathlib import Path
sys.path.insert(0, '..')

from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from statsmodels.stats.multitest import multipletests

from src.rq2_panel import build_rq2_panel, WEAPON_CLASSES, OUTCOMES
from src.io_utils import load_checkpoint, save_checkpoint
from src.config import CLEAN_DIR, FIGURES_DIR, TABLES_DIR, DATA_DIR, SEED

# loky workers import project modules by reference — expose the project root
PROJECT_ROOT = Path("..").resolve()
os.environ["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

FIG_DIR = FIGURES_DIR / "nb13"
TBL_DIR = TABLES_DIR / "nb13"
CACHE_DIR = DATA_DIR / "interim"
for d in (FIG_DIR, TBL_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

panel = build_rq2_panel()
assert panel["iso3"].nunique() == 192, f"expected 192 countries, got {panel['iso3'].nunique()}"

CELLS = [(w, o) for w in WEAPON_CLASSES for o in OUTCOMES]   # same ordering convention as NB12's LAG1_CELLS
assert len(CELLS) == 12, f"expected 12 cells, got {len(CELLS)}"
print(f"CELLS ({len(CELLS)}):")
for c in CELLS:
    print(f"  {c}")

# ── Locked constants (define once, reuse everywhere) ──────────────────────
ALPHA = 0.05
ORIGIN_START = 2005      # first forecast ORIGIN year (train on <= this, forecast this+1)
ORIGIN_END = 2023        # last origin (forecasts through 2024)
MIN_TRAIN_OBS = 200       # floor on pooled training rows before a cell's first origin is usable
N_PERM_CS = 2000          # circular-shift permutations — matches NB12 for methodological consistency
SEED_OFFSET_NB13 = 900_000   # fresh seed-offset block, disjoint from NB12's 500000+i / 700000+i

N_ORIGINS = ORIGIN_END - ORIGIN_START + 1
print(f"\nALPHA = {ALPHA}")
print(f"ORIGIN_START = {ORIGIN_START}, ORIGIN_END = {ORIGIN_END}  "
      f"({N_ORIGINS} origins, forecasting years {ORIGIN_START + 1}-{ORIGIN_END + 1})")
print(f"MIN_TRAIN_OBS = {MIN_TRAIN_OBS}")
print(f"N_PERM_CS = {N_PERM_CS}")
print(f"SEED = {SEED} (src.config)")
print(f"SEED_OFFSET_NB13 = {SEED_OFFSET_NB13}")

print(f"\nPanel: {panel.shape}, {panel['iso3'].nunique()} countries, "
      f"{panel['year'].min()}-{panel['year'].max()}")
print("[Section 0] Setup complete.")


[checkpoint] loaded ← master_panel.parquet  (15,168 rows)
[rq2_panel] 6,912 rows, 192 countries, 1989–2024
CELLS (12):
  ('AIR', 'part_n_minor')
  ('AIR', 'part_n_war')
  ('AIR', 'part_n_extraterritorial')
  ('MISSILES', 'part_n_minor')
  ('MISSILES', 'part_n_war')
  ('MISSILES', 'part_n_extraterritorial')
  ('NAVAL', 'part_n_minor')
  ('NAVAL', 'part_n_war')
  ('NAVAL', 'part_n_extraterritorial')
  ('GROUND', 'part_n_minor')
  ('GROUND', 'part_n_war')
  ('GROUND', 'part_n_extraterritorial')

ALPHA = 0.05
ORIGIN_START = 2005, ORIGIN_END = 2023  (19 origins, forecasting years 2006-2024)
MIN_TRAIN_OBS = 200
N_PERM_CS = 2000
SEED = 42 (src.config)
SEED_OFFSET_NB13 = 900000

Panel: (6912, 30), 192 countries, 1989-2024
[Section 0] Setup complete.


## Section 1 — Rolling-Origin Forecast Loop

**Chosen simplification (stated explicitly, not an oversight):** all four models are fit **pooled** across countries, with no country fixed effects. An expanding training window that starts at ~200 pooled rows (well under 200 country-years worth of history) cannot support ~190 per-country dummies without immediately overfitting on the earliest origins — pooled OLS is the only estimator that stays estimable across the whole 2005–2023 origin range.

In [2]:
def shift_within_group(arr, group_idx_list):
    """Positional shift-by-1 within each country's (already time-sorted) index
    block. First observation of every country becomes NaN — same convention as
    the `{col}_lag1` columns used in src.stats_panel / NB05+."""
    out = np.full(arr.shape, np.nan)
    for idx in group_idx_list:
        if idx.size > 1:
            out[idx[1:]] = arr[idx[:-1]]
    return out


def circular_shift_treat(treat, group_idx_list, rng):
    """Circular-shift permutation — identical rotation procedure to NB12
    Section 2f (src/stats_panel.py companion logic, replicated here since it
    lives inline in NB12's notebook, not in a shared module). Each country's
    FINITE treatment values are rotated by a random non-zero offset; the NaN
    mask (positions, not values) is preserved exactly, so the treatment's own
    autocorrelation structure survives while its alignment with the outcome
    is destroyed."""
    shifted = treat.copy()
    for idx in group_idx_list:
        vals = treat[idx]
        valid = np.isfinite(vals)
        nv = int(valid.sum())
        if nv > 2:
            k = int(rng.integers(1, nv))
            rolled = vals.copy()
            rolled[valid] = np.roll(vals[valid], k)
            shifted[idx] = rolled
    return shifted


def prepare_cell(panel, weapon, outcome):
    """Per-cell numpy arrays + per-origin train/test index sets. Index sets
    are permutation-invariant (circular shift preserves the NaN mask), so
    Section 3 reuses them unchanged and only re-derives treat_lag1 values."""
    treat_col = f"d_log_tiv_{weapon}"
    out_col = f"d_log_{outcome}"
    df = panel[["iso3", "year", treat_col, out_col]].sort_values(["iso3", "year"]).reset_index(drop=True)

    iso3_arr = df["iso3"].to_numpy()
    years = df["year"].to_numpy()
    treat = df[treat_col].to_numpy(dtype=float)
    out_arr = df[out_col].to_numpy(dtype=float)
    base_valid = np.isfinite(treat) & np.isfinite(out_arr)

    group_idx_list = list(df.groupby("iso3").indices.values())
    out_lag1 = shift_within_group(out_arr, group_idx_list)
    treat_lag1 = shift_within_group(treat, group_idx_list)

    origins_info = []
    skipped = []
    for tau in range(ORIGIN_START, ORIGIN_END + 1):
        train_mask = (years <= tau) & base_valid
        test_mask = (years == tau + 1) & base_valid
        train_idx = np.where(train_mask)[0]
        test_idx = np.where(test_mask)[0]

        if train_idx.size < MIN_TRAIN_OBS or test_idx.size == 0:
            reason = (f"insufficient training obs ({train_idx.size} < {MIN_TRAIN_OBS})"
                      if train_idx.size < MIN_TRAIN_OBS else "no test observations")
            skipped.append({"weapon": weapon, "outcome": outcome, "origin": tau,
                            "reason": reason, "n_train": int(train_idx.size),
                            "n_test": int(test_idx.size)})
            continue

        m2_train_finite = np.isfinite(out_lag1[train_idx])
        m3_train_finite = m2_train_finite & np.isfinite(treat_lag1[train_idx])
        m2_test_finite = np.isfinite(out_lag1[test_idx])
        m3_test_finite = m2_test_finite & np.isfinite(treat_lag1[test_idx])

        origins_info.append({
            "origin": tau, "train_idx": train_idx, "test_idx": test_idx,
            "train_m2_idx": train_idx[m2_train_finite], "test_m2_idx": test_idx[m2_test_finite],
            "train_m3_idx": train_idx[m3_train_finite], "test_m3_idx": test_idx[m3_test_finite],
        })

    return {"weapon": weapon, "outcome": outcome, "treat_col": treat_col, "out_col": out_col,
            "iso3": iso3_arr, "years": years, "treat": treat, "out": out_arr,
            "out_lag1": out_lag1, "treat_lag1": treat_lag1, "group_idx_list": group_idx_list,
            "origins_info": origins_info, "skipped": skipped}


CELL_PREP = {(w, o): prepare_cell(panel, w, o) for w, o in CELLS}
print(f"[Section 1] Precomputed per-origin index sets for {len(CELL_PREP)} cells.")


[Section 1] Precomputed per-origin index sets for 12 cells.


In [3]:
def fit_predict_all_models(prep):
    """Fit M0/M1/M2/M3 at every usable origin and return one long DataFrame of
    per-country-year predictions for this cell."""
    out_arr = prep["out"]; out_lag1 = prep["out_lag1"]; treat_lag1 = prep["treat_lag1"]
    iso3_arr = prep["iso3"]
    frames = []
    for info in prep["origins_info"]:
        tau = info["origin"]
        train_idx, test_idx = info["train_idx"], info["test_idx"]
        y_train_full = out_arr[train_idx]
        y_test = out_arr[test_idx]
        iso_test = iso3_arr[test_idx]

        yhat_m0 = np.zeros(test_idx.size)
        yhat_m1 = np.full(test_idx.size, y_train_full.mean())

        tr2, te2 = info["train_m2_idx"], info["test_m2_idx"]
        yhat_m2 = np.full(test_idx.size, np.nan)
        if tr2.size >= 2 and te2.size > 0:
            X2 = np.column_stack([np.ones(tr2.size), out_lag1[tr2]])
            coef2, *_ = np.linalg.lstsq(X2, out_arr[tr2], rcond=None)
            Xp2 = np.column_stack([np.ones(te2.size), out_lag1[te2]])
            yhat_m2[np.isin(test_idx, te2)] = Xp2 @ coef2

        tr3, te3 = info["train_m3_idx"], info["test_m3_idx"]
        yhat_m3 = np.full(test_idx.size, np.nan)
        if tr3.size >= 3 and te3.size > 0:
            X3 = np.column_stack([np.ones(tr3.size), out_lag1[tr3], treat_lag1[tr3]])
            coef3, *_ = np.linalg.lstsq(X3, out_arr[tr3], rcond=None)
            Xp3 = np.column_stack([np.ones(te3.size), out_lag1[te3], treat_lag1[te3]])
            yhat_m3[np.isin(test_idx, te3)] = Xp3 @ coef3

        base = pd.DataFrame({"weapon": prep["weapon"], "outcome": prep["outcome"],
                             "origin": tau, "iso3": iso_test, "y_true": y_test})
        for model_name, yhat in [("M0", yhat_m0), ("M1", yhat_m1), ("M2", yhat_m2), ("M3", yhat_m3)]:
            f = base.copy()
            f["model"] = model_name
            f["y_pred"] = yhat
            f["sq_err"] = (f["y_true"] - f["y_pred"]) ** 2
            f["abs_err"] = (f["y_true"] - f["y_pred"]).abs()
            frames.append(f)
    return pd.concat(frames, ignore_index=True)


t0 = time.perf_counter()
all_frames = [fit_predict_all_models(CELL_PREP[c]) for c in CELLS]
forecast_results_df = pd.concat(all_frames, ignore_index=True)[
    ["weapon", "outcome", "origin", "iso3", "model", "y_true", "y_pred", "sq_err", "abs_err"]]

skipped_log = pd.DataFrame([s for c in CELLS for s in CELL_PREP[c]["skipped"]])

FORECAST_CACHE = CACHE_DIR / "nb13_forecast_results.parquet"
save_checkpoint(forecast_results_df, FORECAST_CACHE)

print(f"\n[Section 1] Rolling-origin loop done in {time.perf_counter() - t0:.1f}s "
      f"-> {len(forecast_results_df):,} prediction rows across {forecast_results_df['weapon'].nunique()} "
      f"weapons x {forecast_results_df['outcome'].nunique()} outcomes x "
      f"{forecast_results_df['model'].nunique()} models.")
print(f"Skipped origins: {len(skipped_log)} "
      f"(0 expected here since MIN_TRAIN_OBS={MIN_TRAIN_OBS} clears easily by "
      f"ORIGIN_START={ORIGIN_START})")
if len(skipped_log):
    print(skipped_log.to_string(index=False))


[checkpoint] saved → nb13_forecast_results.parquet  (175,104 rows)

[Section 1] Rolling-origin loop done in 2.2s -> 175,104 prediction rows across 4 weapons x 3 outcomes x 4 models.
Skipped origins: 0 (0 expected here since MIN_TRAIN_OBS=200 clears easily by ORIGIN_START=2005)


## Section 2 — Pooled OOS Skill Metrics

Skill is reported against **M2** (AR(1)-only) — the natural baseline, since M2 already knows the outcome's own recent history. M0/M1 are reported as a floor, not the main comparison.

In [4]:
def pooled_rmse_mae(df, model):
    sub = df.loc[df["model"] == model, ["sq_err", "abs_err"]].dropna()
    if len(sub) == 0:
        return np.nan, np.nan
    return float(np.sqrt(sub["sq_err"].mean())), float(sub["abs_err"].mean())


rows = []
for w, o in CELLS:
    sub = forecast_results_df[(forecast_results_df["weapon"] == w) & (forecast_results_df["outcome"] == o)]
    rec = {"weapon": w, "outcome": o}
    for m in ["M0", "M1", "M2", "M3"]:
        rmse, mae = pooled_rmse_mae(sub, m)
        rec[f"rmse_{m}"] = rmse
        rec[f"mae_{m}"] = mae
    rec["skill_M3_vs_M2"] = 1 - rec["rmse_M3"] / rec["rmse_M2"]
    rec["n_forecasts"] = int((sub["model"] == "M0").sum())
    rec["n_origins_used"] = len(CELL_PREP[(w, o)]["origins_info"])
    rec["n_origins_skipped"] = len(CELL_PREP[(w, o)]["skipped"])
    rows.append(rec)

section2_df = pd.DataFrame(rows)
section2_df.to_csv(TBL_DIR / "section2_skill_by_cell.csv", index=False)

print("=== Pooled OOS skill by cell (vs M2 AR(1)-only baseline) ===")
print(section2_df[["weapon", "outcome", "rmse_M0", "rmse_M1", "rmse_M2", "rmse_M3",
                   "skill_M3_vs_M2", "n_forecasts", "n_origins_used",
                   "n_origins_skipped"]].to_string(index=False))
print(f"\n[Section 2] Saved -> section2_skill_by_cell.csv (12 rows)")


=== Pooled OOS skill by cell (vs M2 AR(1)-only baseline) ===
  weapon                 outcome  rmse_M0  rmse_M1  rmse_M2  rmse_M3  skill_M3_vs_M2  n_forecasts  n_origins_used  n_origins_skipped
     AIR            part_n_minor 0.395197 0.396350 0.365884 0.366089       -0.000561         3648              19                  0
     AIR              part_n_war 0.222600 0.223122 0.214067 0.214049        0.000084         3648              19                  0
     AIR part_n_extraterritorial 0.345510 0.346390 0.336397 0.336492       -0.000284         3648              19                  0
MISSILES            part_n_minor 0.395197 0.396350 0.365884 0.365961       -0.000211         3648              19                  0
MISSILES              part_n_war 0.222600 0.223122 0.214067 0.214085       -0.000083         3648              19                  0
MISSILES part_n_extraterritorial 0.345510 0.346390 0.336397 0.336452       -0.000166         3648              19                  0
   NAVAL

## Section 3 — Circular-Shift Significance Test on the Skill Differential

Reuses NB12's circular-shift null (Section 2f) rather than a plain Diebold-Mariano test — NB12 already established that a naive shuffle destroys the treatment's negative serial structure and is the wrong null here. M2 is unchanged across permutations (it never touches the treatment); only M3's treatment predictor is circularly shifted, refit, and re-forecast at every origin.

In [5]:
def run_cs_perm_task(perm_idx, weapon, outcome, treat, out_arr, out_lag1,
                      group_idx_list, origins_info, seed_base):
    rng = np.random.default_rng([SEED, seed_base + perm_idx])
    shifted_treat = circular_shift_treat(treat, group_idx_list, rng)
    shifted_treat_lag1 = shift_within_group(shifted_treat, group_idx_list)

    sse, n = 0.0, 0
    for info in origins_info:
        tr, te = info["train_m3_idx"], info["test_m3_idx"]
        if tr.size < 3 or te.size == 0:
            continue
        X_train = np.column_stack([np.ones(tr.size), out_lag1[tr], shifted_treat_lag1[tr]])
        coef, *_ = np.linalg.lstsq(X_train, out_arr[tr], rcond=None)
        X_test = np.column_stack([np.ones(te.size), out_lag1[te], shifted_treat_lag1[te]])
        yhat = X_test @ coef
        sse += float(np.sum((out_arr[te] - yhat) ** 2))
        n += te.size
    rmse_perm = np.sqrt(sse / n) if n > 0 else np.nan
    return {"weapon": weapon, "outcome": outcome, "perm": perm_idx, "rmse_m3_perm": rmse_perm}


CS_CACHE = CACHE_DIR / f"nb13_circshift_skill_{N_PERM_CS}.parquet"
expected_rows_cs = N_PERM_CS * len(CELLS)

cache_ok = False
try:
    circshift_df = load_checkpoint(CS_CACHE)
    cache_ok = (len(circshift_df) == expected_rows_cs
                and circshift_df["perm"].nunique() == N_PERM_CS
                and set(zip(circshift_df["weapon"], circshift_df["outcome"])) == set(CELLS))
except FileNotFoundError:
    cache_ok = False

if cache_ok:
    print(f"Cache HIT — loaded {len(circshift_df):,} circular-shift skill draws from {CS_CACHE.name}")
else:
    print(f"Cache MISS — running {N_PERM_CS:,} circular-shift permutations x {len(CELLS)} cells "
          f"= {expected_rows_cs:,} full rolling-origin M3 refits ...")
    t0 = time.perf_counter()
    all_results = []
    for cell_idx, (w, o) in enumerate(CELLS):
        prep = CELL_PREP[(w, o)]
        seed_base = SEED_OFFSET_NB13 + cell_idx * 10_000
        cell_results = Parallel(n_jobs=-2)(
            delayed(run_cs_perm_task)(i, w, o, prep["treat"], prep["out"], prep["out_lag1"],
                                      prep["group_idx_list"], prep["origins_info"], seed_base)
            for i in range(N_PERM_CS))
        all_results.extend(cell_results)
        print(f"  [{cell_idx + 1}/{len(CELLS)}] {w}x{o} done "
              f"({time.perf_counter() - t0:.0f}s elapsed)")
    circshift_df = pd.DataFrame(all_results)
    save_checkpoint(circshift_df, CS_CACHE)
    print(f"Computed fresh in {time.perf_counter() - t0:.0f}s -> cached to {CS_CACHE.name}")

print(f"\n[Section 3] Circular-shift cache: {len(circshift_df):,} rows "
      f"({N_PERM_CS:,} perms x {len(CELLS)} cells)")


[checkpoint] loaded ← nb13_circshift_skill_2000.parquet  (24,000 rows)
Cache HIT — loaded 24,000 circular-shift skill draws from nb13_circshift_skill_2000.parquet

[Section 3] Circular-shift cache: 24,000 rows (2,000 perms x 12 cells)


In [6]:
rmse_m2_observed = section2_df.set_index(["weapon", "outcome"])["rmse_M2"]
circshift_df["skill_m3_perm"] = 1 - circshift_df["rmse_m3_perm"] / circshift_df.apply(
    lambda r: rmse_m2_observed.loc[(r["weapon"], r["outcome"])], axis=1)

perm_rows = []
for w, o in CELLS:
    obs_skill = float(section2_df.loc[(section2_df["weapon"] == w) & (section2_df["outcome"] == o),
                                       "skill_M3_vs_M2"].iloc[0])
    null = circshift_df.loc[(circshift_df["weapon"] == w) & (circshift_df["outcome"] == o),
                            "skill_m3_perm"].to_numpy()
    null_f = null[np.isfinite(null)]
    n_exceed = int((null_f >= obs_skill).sum())
    p_cell = (1 + n_exceed) / (1 + N_PERM_CS)
    perm_rows.append({
        "weapon": w, "outcome": o, "skill_M3_vs_M2": round(obs_skill, 5),
        "null_mean_skill": round(float(np.mean(null_f)), 5) if null_f.size else np.nan,
        "null_sd_skill": round(float(np.std(null_f)), 5) if null_f.size else np.nan,
        "null_p95_skill": round(float(np.percentile(null_f, 95)), 5) if null_f.size else np.nan,
        "n_finite_perm": int(null_f.size), "n_exceed": n_exceed,
        "p_cell": round(p_cell, 5),
    })
section3_df = pd.DataFrame(perm_rows)
section3_df["q_bh"] = np.round(
    multipletests(section3_df["p_cell"].values, alpha=ALPHA, method="fdr_bh")[1], 5)
section3_df["survives_nominal"] = section3_df["p_cell"] < ALPHA
section3_df["survives_bh"] = section3_df["q_bh"] < ALPHA
section3_df = section3_df.sort_values("p_cell").reset_index(drop=True)

section3_df.to_csv(TBL_DIR / "section3_percell_permutation.csv", index=False)

print("=== Circular-shift permutation test on skill_M3_vs_M2, per cell ===")
print(section3_df[["weapon", "outcome", "skill_M3_vs_M2", "null_mean_skill", "null_p95_skill",
                   "p_cell", "q_bh", "survives_nominal", "survives_bh"]].to_string(index=False))
n_survive_nominal = int(section3_df["survives_nominal"].sum())
n_survive_bh = int(section3_df["survives_bh"].sum())
print(f"\n{n_survive_nominal}/12 cells nominally significant (p<{ALPHA}); "
      f"{n_survive_bh}/12 survive BH (smallest q={section3_df['q_bh'].min():.4f})")
print(f"\n[Section 3] Saved -> section3_percell_permutation.csv")


=== Circular-shift permutation test on skill_M3_vs_M2, per cell ===
  weapon                 outcome  skill_M3_vs_M2  null_mean_skill  null_p95_skill  p_cell    q_bh  survives_nominal  survives_bh
  GROUND              part_n_war         0.00021         -0.00018         0.00033 0.08646 0.70864             False        False
     AIR              part_n_war         0.00008         -0.00016         0.00028 0.13493 0.70864             False        False
   NAVAL part_n_extraterritorial        -0.00001         -0.00007         0.00024 0.27086 0.70864             False        False
   NAVAL            part_n_minor        -0.00004         -0.00007         0.00029 0.31684 0.70864             False        False
MISSILES              part_n_war        -0.00008         -0.00017         0.00028 0.34783 0.70864             False        False
  GROUND part_n_extraterritorial        -0.00006         -0.00007         0.00026 0.35432 0.70864             False        False
   NAVAL              part_n_

## Section 4 — Figures

In [7]:
fig, ax = plt.subplots(figsize=(11, 5.5))
weapons = list(WEAPON_CLASSES)
outcomes = OUTCOMES
n_out = len(outcomes)
bar_w = 0.8 / n_out
colors = {"part_n_minor": "steelblue", "part_n_war": "indianred", "part_n_extraterritorial": "seagreen"}

for j, o in enumerate(outcomes):
    xs, heights, null_p95 = [], [], []
    for i, w in enumerate(weapons):
        x = i + (j - (n_out - 1) / 2) * bar_w
        row2 = section2_df[(section2_df["weapon"] == w) & (section2_df["outcome"] == o)].iloc[0]
        row3 = section3_df[(section3_df["weapon"] == w) & (section3_df["outcome"] == o)].iloc[0]
        xs.append(x)
        heights.append(row2["skill_M3_vs_M2"])
        null_p95.append(row3["null_p95_skill"])
    ax.bar(xs, heights, width=bar_w * 0.95, color=colors[o], label=o, zorder=3)
    ax.scatter(xs, null_p95, marker="_", s=260, color="black", linewidths=1.6, zorder=4,
              label="null 95th pct" if j == 0 else None)

ax.axhline(0, color="gray", lw=0.8)
ax.set_xticks(range(len(weapons)))
ax.set_xticklabels(weapons)
ax.set_ylabel("skill_M3_vs_M2 = 1 - RMSE(M3)/RMSE(M2)", fontsize=9)
ax.set_title("OOS forecast skill of treatment-augmented M3 vs AR(1)-only M2, by cell", fontsize=10)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_skill_by_cell.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("[Section 4] Saved -> fig1_skill_by_cell.png")


[Section 4] Saved -> fig1_skill_by_cell.png


In [8]:
best_row = section2_df.loc[section2_df["skill_M3_vs_M2"].idxmax()]
best_w, best_o = best_row["weapon"], best_row["outcome"]
best_sub = forecast_results_df[(forecast_results_df["weapon"] == best_w)
                               & (forecast_results_df["outcome"] == best_o)
                               & (forecast_results_df["model"] == "M3")].dropna(subset=["y_pred"])

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(best_sub["y_true"], best_sub["y_pred"], s=14, alpha=0.5, color="steelblue")
lims = [min(best_sub["y_true"].min(), best_sub["y_pred"].min()),
        max(best_sub["y_true"].max(), best_sub["y_pred"].max())]
ax.plot(lims, lims, color="indianred", lw=1.2, ls="--", label="y = x")
ax.set_xlabel(f"actual d_log_{best_o}", fontsize=9)
ax.set_ylabel(f"predicted d_log_{best_o} (M3)", fontsize=9)
ax.set_title(f"Highest-skill cell: {best_w}x{best_o}  "
            f"(skill_M3_vs_M2={best_row['skill_M3_vs_M2']:.3f})", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_forecast_vs_actual_example.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[Section 4] Highest-skill cell: {best_w}x{best_o} -> fig2_forecast_vs_actual_example.png")


[Section 4] Highest-skill cell: GROUNDxpart_n_war -> fig2_forecast_vs_actual_example.png


## Section 5 — Sanity Checks

In [9]:
checks = []

# [1] All 12 cells present
_cells_present = set(zip(forecast_results_df["weapon"], forecast_results_df["outcome"]))
checks.append(("All 12 cells present in forecast_results_df", _cells_present == set(CELLS)))

# [2] n_origins_used + n_origins_skipped == N_ORIGINS for every cell
_ok2 = bool(((section2_df["n_origins_used"] + section2_df["n_origins_skipped"]) == N_ORIGINS).all())
checks.append((f"n_origins_used + n_origins_skipped == {N_ORIGINS} for every cell", _ok2))

# [3] every skipped origin has a reason logged
_ok3 = (len(skipped_log) == 0) or bool(skipped_log["reason"].apply(lambda r: len(str(r)) > 0).all())
checks.append((f"Every skipped origin has a reason logged ({len(skipped_log)} skipped total)", _ok3))

# [4] RMSE(M0) vs RMSE(M1) — printed for inspection, NOT asserted
print("=== [4] RMSE(M0) vs RMSE(M1), printed for inspection only (no ordering asserted) ===")
print(section2_df[["weapon", "outcome", "rmse_M0", "rmse_M1"]].to_string(index=False))
checks.append(("RMSE(M0) vs RMSE(M1) printed for inspection (no assertion)", True))

# [5] circular-shift cache has exactly N_PERM_CS x 12 rows
checks.append((f"Circular-shift cache has exactly {N_PERM_CS} x 12 = {N_PERM_CS * 12} rows",
               len(circshift_df) == N_PERM_CS * 12))

# [6] p_cell in [0,1], one-sided upper tail
_p_ok = section3_df["p_cell"].between(0, 1).all()
print("\n[6] p_cell is one-sided upper-tail: p = (1 + #{perm skill >= observed skill}) / (1 + N_PERM_CS) "
      "— only permutations whose skill matches or beats the observed skill count against it.")
checks.append(("p_cell in [0,1] for all 12 cells, one-sided upper tail", bool(_p_ok)))

# [7] BH q >= p (monotonicity sanity)
_ok7 = bool((section3_df["q_bh"] >= section3_df["p_cell"] - 1e-6).all())
checks.append(("BH q >= p for all cells (monotonicity sanity)", _ok7))

# [8] all tables + figures saved
_expected = [TBL_DIR / "section2_skill_by_cell.csv", TBL_DIR / "section3_percell_permutation.csv",
            FIG_DIR / "fig1_skill_by_cell.png", FIG_DIR / "fig2_forecast_vs_actual_example.png",
            FORECAST_CACHE, CS_CACHE]
_missing = [p.name for p in _expected if not p.exists()]
print(f"\n[8] Expected outputs: {[p.name for p in _expected]}")
checks.append((f"All tables + figures + caches saved (missing: {_missing or 'none'})", len(_missing) == 0))

# [9] seed streams documented and disjoint
_nb12_blocks = [(500_000, 500_000 + 2000), (700_000, 700_000 + 2000)]
_nb13_block = (SEED_OFFSET_NB13, SEED_OFFSET_NB13 + len(CELLS) * 10_000)
_disjoint = all(_nb13_block[0] >= hi or _nb13_block[1] <= lo for lo, hi in _nb12_blocks)
print(f"\n[9] Forecast-loop randomness: NONE (OLS via np.linalg.lstsq is deterministic).")
print(f"    Permutation seed stream: [SEED, {SEED_OFFSET_NB13} + cell_idx*10000 + perm_idx], "
      f"cell_idx in 0..{len(CELLS)-1}, perm_idx in 0..{N_PERM_CS-1} "
      f"-> range [{_nb13_block[0]}, {_nb13_block[1]})")
print(f"    NB12 used [500000, 502000) and [700000, 702000) — disjoint from NB13's range: {_disjoint}")
checks.append(("Seed stream documented and disjoint from NB12's 500000+i / 700000+i", _disjoint))

print("\n=== NB13 sanity checks ===\n")
n_pass = 0
for i, (label, ok) in enumerate(checks, 1):
    status = "PASS" if ok else "FAIL"
    n_pass += int(ok)
    print(f"[{i}] {status} — {label}")
print(f"\n{n_pass}/{len(checks)} checks passed")


=== [4] RMSE(M0) vs RMSE(M1), printed for inspection only (no ordering asserted) ===
  weapon                 outcome  rmse_M0  rmse_M1
     AIR            part_n_minor 0.395197 0.396350
     AIR              part_n_war 0.222600 0.223122
     AIR part_n_extraterritorial 0.345510 0.346390
MISSILES            part_n_minor 0.395197 0.396350
MISSILES              part_n_war 0.222600 0.223122
MISSILES part_n_extraterritorial 0.345510 0.346390
   NAVAL            part_n_minor 0.395197 0.396350
   NAVAL              part_n_war 0.222600 0.223122
   NAVAL part_n_extraterritorial 0.345510 0.346390
  GROUND            part_n_minor 0.395197 0.396350
  GROUND              part_n_war 0.222600 0.223122
  GROUND part_n_extraterritorial 0.345510 0.346390

[6] p_cell is one-sided upper-tail: p = (1 + #{perm skill >= observed skill}) / (1 + N_PERM_CS) — only permutations whose skill matches or beats the observed skill count against it.

[8] Expected outputs: ['section2_skill_by_cell.csv', 'section3_perce

## Section 6 — Headline Findings

In [10]:
print("=" * 74)
print("NB13 HEADLINE FINDINGS — does knowing weapon acquisitions genuinely improve")
print("out-of-sample conflict forecasts?")
print("=" * 74)

_survivors = section3_df[(section3_df["skill_M3_vs_M2"] > 0)
                         & section3_df["survives_nominal"] & section3_df["survives_bh"]]

print("\n1. Cells with skill_M3_vs_M2 > 0 AND p_cell < 0.05 (nominal) AND surviving BH:")
if len(_survivors):
    print(_survivors[["weapon", "outcome", "skill_M3_vs_M2", "p_cell", "q_bh"]].to_string(index=False))
else:
    print("   None.")

_any_bh = bool(section3_df["survives_bh"].any())
print(f"\n2. Does ANY cell survive BH? {'YES' if _any_bh else 'NO'} "
      f"({int(section3_df['survives_bh'].sum())}/12; smallest q_bh={section3_df['q_bh'].min():.4f})")

_best = section2_df.loc[section2_df["skill_M3_vs_M2"].idxmax()]
_worst = section2_df.loc[section2_df["skill_M3_vs_M2"].idxmin()]
_n_pos = int((section2_df["skill_M3_vs_M2"] > 0).sum())
print("\n3. Connecting back to NB12:")
_agreement = (
    f"The forecasting benchmark {'CORROBORATES' if not _any_bh else 'DIVERGES FROM'} "
    f"NB12's inferential null. {_n_pos}/12 cells show positive raw OOS skill "
    f"(best: {_best['weapon']}x{_best['outcome']} at {_best['skill_M3_vs_M2']:.4f}; "
    f"worst: {_worst['weapon']}x{_worst['outcome']} at {_worst['skill_M3_vs_M2']:.4f}), but "
    f"{'none of that skill' if not _any_bh else 'some of that skill'} clears its own "
    f"circular-shift null after BH correction across the same 12-cell family NB12 used. "
    f"{'This agrees with NB12 at the level that matters for the paper: a genuinely '
      'held-out forecasting test, using a different estimator (pooled OLS) and a '
      'different loss (RMSE) than the DH panel-Granger pipeline, reaches the same '
      'conclusion — no weapon-class cell offers forecast-usable information about '
      'next-year conflict participation beyond the AR(1) history of the outcome itself.'
      if not _any_bh else
      'This is a genuine divergence from NB12: at least one cell forecasts better '
      'out-of-sample than chance despite not surviving the DH lead-lag test, worth '
      'flagging explicitly in the paper as a case where predictive skill and '
      'statistical significance disagree.'}"
)
print(f"   {_agreement}")

print()
print("=== NB-13 complete — proceed to NB-14 (naive vs. rigorous comparison) ===")


NB13 HEADLINE FINDINGS — does knowing weapon acquisitions genuinely improve
out-of-sample conflict forecasts?

1. Cells with skill_M3_vs_M2 > 0 AND p_cell < 0.05 (nominal) AND surviving BH:
   None.

2. Does ANY cell survive BH? NO (0/12; smallest q_bh=0.7086)

3. Connecting back to NB12:
   The forecasting benchmark CORROBORATES NB12's inferential null. 2/12 cells show positive raw OOS skill (best: GROUNDxpart_n_war at 0.0002; worst: AIRxpart_n_minor at -0.0006), but none of that skill clears its own circular-shift null after BH correction across the same 12-cell family NB12 used. This agrees with NB12 at the level that matters for the paper: a genuinely held-out forecasting test, using a different estimator (pooled OLS) and a different loss (RMSE) than the DH panel-Granger pipeline, reaches the same conclusion — no weapon-class cell offers forecast-usable information about next-year conflict participation beyond the AR(1) history of the outcome itself.

=== NB-13 complete — proceed t